# カラー図形を学ぶCNN / A CNN that learns colored shapes
Google Colabで **ランタイム → すべてのセルを実行**。GPU不要。 / In Google Colab, choose **Runtime → Run all**. No GPU required.

**円・三角・四角を形で分類します。色は正解と独立にランダム化。位置・大きさ・向き・背景ノイズも変えます。** / Classify circles, triangles, and squares by shape. Color is randomized independently of the label, along with position, size, orientation, and background noise.

データはプログラム内で生成する16×16のRGB画像。外部ダウンロード不要。 / Synthetic 16×16 RGB images, generated locally without downloads.

**16×16×3 → Conv(8)+ReLU → MaxPool → Conv(16)+ReLU → MaxPool → Flatten(256) → Linear(3)+Softmax**

Convは3×3、stride 1、padding 1。Poolは2×2、stride 2。 / Conv: 3×3, stride 1, padding 1. Pool: 2×2, stride 2.

1. 学習と汎化を確認 / Verify learning and generalization
2. 学習前後の特徴マップを比較 / Compare untrained and trained maps
3. 同じ画像を移動させて応答を比較 / Translate an image and compare responses
4. RGBをまとめるフィルタと積和を確認 / Inspect filters and dot products across RGB

NumPyで順伝播・逆伝播・Adamを実装。授業用の小規模デモです。 / Forward/backward passes and Adam use NumPy. This is a small teaching demo.

In [ ]:
# ライブラリと表示設定 / Imports and display setup
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display, clear_output
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
# 小規模演算のスレッド過剰使用を防ぐ / Limit thread overhead for small operations
from threadpoolctl import threadpool_limits
_cpu_limits = threadpool_limits(limits=1)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

## 1. 図形データの生成と分割 / Generate and split shapes
アンチエイリアス処理をして16×16に縮小します。どのクラスも同じ色分布です。 / Shapes are antialiased by downsampling. All classes share the same color distribution.

60%訓練、20%検証、20%テスト。検証画像だけを観察UIに使います。 / 60% training, 20% validation, 20% test. The explorer uses validation images only.

16×16でも、画像は平坦化せず **バッチ×RGB×高さ×幅** で入力します。 / Images enter as **batch × RGB × height × width**, without flattening.

In [ ]:
# 図形を描く / Render randomized shapes
CLASS_NAMES = ['Circle', 'Triangle', 'Square']
SIDE = 16
N_PER_CLASS = 800

def make_shape(label, rng):
    # 高解像度で描いて平均縮小 / Render at higher resolution and average down
    scale=4
    yy,xx=np.mgrid[0:SIDE*scale,0:SIDE*scale].astype(np.float32)
    xx=(xx+.5)/scale; yy=(yy+.5)/scale
    cx,cy=rng.uniform(5.8,10.2,2); radius=rng.uniform(2.5,3.7)
    angle=rng.uniform(-np.pi,np.pi)
    dx=xx-cx; dy=yy-cy
    u=np.cos(angle)*dx+np.sin(angle)*dy
    v=-np.sin(angle)*dx+np.cos(angle)*dy
    if label==0:
        mask=(u*u+v*v)<=radius**2
    elif label==1:
        # 正三角形の３辺による判定 / Test the three half-planes of an equilateral triangle
        triangle_radius=radius*np.sqrt(4*np.pi/(3*np.sqrt(3)))
        radius=triangle_radius
        vertices=np.array([[0,-radius],[np.sqrt(3)*radius/2,radius/2],[-np.sqrt(3)*radius/2,radius/2]])
        signs=[]
        for a,b in zip(vertices,np.roll(vertices,-1,axis=0)):
            signs.append((b[0]-a[0])*(v-a[1])-(b[1]-a[1])*(u-a[0]))
        mask=np.all(np.stack(signs)>=0,axis=0)
    else:
        # 円とおおむね同じ面積の正方形 / Square with approximately the circle's area
        half=radius*np.sqrt(np.pi)/2
        mask=(abs(u)<=half)&(abs(v)<=half)
    alpha=mask.reshape(SIDE,scale,SIDE,scale).mean(axis=(1,3))
    color=rng.uniform(.3,1.0,3)
    color[rng.integers(3)]=rng.uniform(.8,1.0)
    bg=rng.uniform(0,.13,3)
    image=bg[:,None,None]*(1-alpha)+color[:,None,None]*alpha
    image+=rng.normal(0,.025,image.shape)
    return np.clip(image,0,1).astype(np.float32)

rng_data=np.random.default_rng(42)
y=np.repeat(np.arange(3),N_PER_CLASS)
X=np.stack([make_shape(int(t),rng_data) for t in y])
Xtrain,Xrest,ytrain,yrest=train_test_split(X,y,test_size=.4,stratify=y,random_state=42)
Xval,Xtest,yval,ytest=train_test_split(Xrest,yrest,test_size=.5,stratify=yrest,random_state=42)
print('Train / Validation / Test:',len(ytrain),len(yval),len(ytest))
fig,axs=plt.subplots(3,7,figsize=(11,5))
for d in range(3):
    for j,ax in enumerate(axs[d]):
        ax.imshow(Xtrain[ytrain==d][j].transpose(1,2,0),interpolation='nearest')
        ax.set_title(CLASS_NAMES[d]); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. CNNの計算 / CNN computations
Convは各出力位置で**近傍×全入力チャネル**の積和を計算します。フィルタ数が出力チャネル数です。 / At each output position, convolution sums products across a neighborhood and all input channels. Filter count determines output channels.

MaxPoolは各チャネルの2×2領域の最大値を残します。学習する重みはありません。 / MaxPool keeps the maximum of each 2×2 region independently per channel. It has no learned weights.

逆伝播では最大値の位置へ勾配を戻します。同値の場合は最初の位置を選びます。 / Backpropagation routes gradients to the selected maximum; ties use the first position.

In [ ]:
# 畳み込みと逆伝播 / Convolution and backward pass
from numpy.lib.stride_tricks import sliding_window_view

def conv_forward(x, w, b):
    p = np.pad(x, ((0,0),(0,0),(1,1),(1,1)))
    patches = sliding_window_view(p, (3,3), axis=(2,3))
    z = np.einsum('nchwkl,ockl->nohw', patches, w, optimize=True)
    return z + b[None,:,None,None], (x.shape, patches, w)

def conv_backward(dz, cache):
    shape, patches, w = cache
    dw = np.einsum('nohw,nchwkl->ockl', dz, patches, optimize=True)
    db = dz.sum(axis=(0,2,3))
    dxpad = np.zeros((shape[0], shape[1], shape[2]+2, shape[3]+2), dtype=dz.dtype)
    for i in range(3):
        for j in range(3):
            dxpad[:,:,i:i+shape[2],j:j+shape[3]] += np.einsum('nohw,oc->nchw', dz, w[:,:,i,j], optimize=True)
    return dxpad[:,:,1:-1,1:-1], dw, db

def pool_forward(x):
    n,c,h,w = x.shape
    blocks = x.reshape(n,c,h//2,2,w//2,2).transpose(0,1,2,4,3,5).reshape(n,c,h//2,w//2,4)
    idx = blocks.argmax(axis=-1)
    return blocks.max(axis=-1), (x.shape, idx)

def pool_backward(dout, cache):
    shape, idx = cache
    g = np.zeros((*dout.shape,4), dtype=dout.dtype)
    np.put_along_axis(g, idx[...,None], dout[...,None], axis=-1)
    n,c,h,w = shape
    return g.reshape(n,c,h//2,w//2,2,2).transpose(0,1,2,4,3,5).reshape(shape)

# He初期化と状態コピー / He initialization and state copies
def initialize(seed=7):
    r = np.random.default_rng(seed)
    return dict(w1=(r.normal(size=(8,3,3,3))*np.sqrt(2/27)).astype(np.float32), b1=np.zeros(8,np.float32),
                w2=(r.normal(size=(16,8,3,3))*np.sqrt(2/72)).astype(np.float32), b2=np.zeros(16,np.float32),
                wf=(r.normal(size=(256,3))*np.sqrt(1/256)).astype(np.float32), bf=np.zeros(3,np.float32))

def copy_params(p):
    return {k:v.copy() for k,v in p.items()}

def forward(x,p):
    z1,c1=conv_forward(x,p['w1'],p['b1']); a1=np.maximum(z1,0)
    q1,k1=pool_forward(a1)
    z2,c2=conv_forward(q1,p['w2'],p['b2']); a2=np.maximum(z2,0)
    q2,k2=pool_forward(a2)
    flat=q2.reshape(len(x),-1)
    logits=flat@p['wf']+p['bf']
    shifted=logits-logits.max(axis=1,keepdims=True)
    exp=np.exp(shifted); prob=exp/exp.sum(axis=1,keepdims=True)
    maps={'Input':x,'Conv1 + ReLU':a1,'Pool1':q1,'Conv2 + ReLU':a2,'Pool2':q2}
    cache=(c1,k1,c2,k2,z1,z2,flat,q2.shape,prob)
    return prob,maps,cache

def loss_grads(x,t,p):
    prob,maps,c=forward(x,p)
    c1,k1,c2,k2,z1,z2,flat,qshape,_=c
    g=prob.copy(); g[np.arange(len(t)),t]-=1; g/=len(t)
    grads={'wf':flat.T@g,'bf':g.sum(axis=0)}
    dq=(g@p['wf'].T).reshape(qshape)
    dz2=pool_backward(dq,k2)*(z2>0)
    dq1,grads['w2'],grads['b2']=conv_backward(dz2,c2)
    dz1=pool_backward(dq1,k1)*(z1>0)
    _,grads['w1'],grads['b1']=conv_backward(dz1,c1)
    return grads

def metrics(x,t,p):
    prob=np.concatenate([forward(x[i:i+128],p)[0] for i in range(0,len(x),128)])
    return -np.log(np.maximum(prob[np.arange(len(t)),t],1e-12)).mean(), (prob.argmax(1)==t).mean()

## 3. 学習する / Train
このセルは実行すると直ちに学習します。Startボタンは不要です。CPUで実行可能ですが所要時間は環境に依存します。 / This cell starts training directly, without a Start button. CPU runtime varies by environment.

各エポックの重みを保存するので、後で学習前後を同じ画像で比較できます。再実行すると最初から学習し直します。 / Saves every epoch for later comparison on the same image. Re-running starts a fresh training run.

In [ ]:
# 調整可能な設定 / Adjustable settings
EPOCHS = 60
LEARNING_RATE = 0.001
BATCH_SIZE = 64

# 初期化とAdam / Initialization and Adam
params=initialize(); snapshots={0:copy_params(params)}
history=[]; rng=np.random.default_rng(123)
m={k:np.zeros_like(v) for k,v in params.items()}
v={k:np.zeros_like(a) for k,a in params.items()}; step=0
for epoch in range(1,EPOCHS+1):
    order=rng.permutation(len(ytrain))
    for start in range(0,len(order),BATCH_SIZE):
        ix=order[start:start+BATCH_SIZE]
        grads=loss_grads(Xtrain[ix],ytrain[ix],params); step+=1
        for k in params:
            m[k]=.9*m[k]+.1*grads[k]; v[k]=.999*v[k]+.001*grads[k]**2
            params[k]-=LEARNING_RATE*(m[k]/(1-.9**step))/(np.sqrt(v[k]/(1-.999**step))+1e-8)
    tr=metrics(Xtrain,ytrain,params); va=metrics(Xval,yval,params)
    history.append((tr[0],va[0],tr[1],va[1])); snapshots[epoch]=copy_params(params)
    if epoch==1 or epoch%5==0 or epoch==EPOCHS:
        print(f'Epoch {epoch:2d}/{EPOCHS}: train={tr[1]:.1%}, val={va[1]:.1%}, val loss={va[0]:.4f}',flush=True)
print('学習完了。下の観察セルへ。 / Training complete. Continue to the explorer.')
fig,axs=plt.subplots(1,2,figsize=(11,3))
h=np.array(history); ep=np.arange(1,EPOCHS+1)
for ax,cols,title in [(axs[0],(0,1),'Cross-entropy'),(axs[1],(2,3),'Accuracy')]:
    for col,label in zip(cols,['Train','Validation']): ax.plot(ep,h[:,col],label=label)
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=.2)
plt.show()

## 4. 学習前後の特徴マップ / Feature maps before and after learning
**Epoch → Shape → Example → Layer** で切り替えます。Epoch 0は学習前。 / Select Epoch, Shape, Example, and Layer. Epoch 0 is untrained.
選択を変更したら **Show maps / 表示** を押してください。 / After changing selections, press **Show maps / 表示**.

同じ画像で最初と最後の応答を比較します。Pool1は16×16→8×8、Pool2は8×8→4×4です。 / Compare checkpoints on the same image. Pool1: 16×16→8×8. Pool2: 8×8→4×4.

同一画面内で共通の色スケールならチャネル間の応答を比較できます。エポックごとにはスケールが変わるので、数値も確認してください。 / Shared scale supports comparison across channels within one view; limits change across epochs, so also inspect numerical values.

特徴は必ずしも「角」などと命名できません。真っ暗なマップは、この入力に対する反応が０という意味です。 / Features need not have simple semantic names. A black map means zero response for this input.

In [ ]:
def select_image(shape,example):
    candidates=np.flatnonzero(yval==shape)
    if not 0<=int(example)<len(candidates):
        raise ValueError('Exampleの範囲外 / Example index out of range')
    return int(candidates[int(example)])

def show_channel_grid(acts,title,shared=True):
    n=len(acts); cols=4; rows=(n+3)//4
    fig,axs=plt.subplots(rows,cols,figsize=(10,2.3*rows),squeeze=False)
    upper=max(float(acts.max()),1e-8)
    for c,ax in enumerate(axs.flat):
        if c<n:
            lim=upper if shared else max(float(acts[c].max()),1e-8)
            im=ax.imshow(acts[c],cmap='viridis',vmin=0,vmax=lim,interpolation='nearest')
            ax.set_title(f'Channel {c} | max {acts[c].max():.2f}')
            fig.colorbar(im,ax=ax,fraction=.046,pad=.04)
        ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

stage_names=['Conv1 + ReLU','Pool1','Conv2 + ReLU','Pool2']
def inspect_maps(epoch=EPOCHS,shape=0,example=0,layer='Conv1 + ReLU',shared=True):
    index=select_image(shape,example); p=snapshots[int(epoch)]
    image=Xval[index:index+1]; prob,maps,_=forward(image,p)
    acts=maps[layer][0]
    fig,axs=plt.subplots(1,2,figsize=(9,2.8))
    axs[0].imshow(image[0].transpose(1,2,0),interpolation='nearest')
    axs[0].set_title(f'True: {CLASS_NAMES[shape]} | Pred: {CLASS_NAMES[prob.argmax()]} | Epoch {epoch}')
    axs[0].axis('off'); axs[1].bar(CLASS_NAMES,prob[0],color='#008c95'); axs[1].set_ylim(0,1)
    axs[1].set_title('Class probabilities'); plt.tight_layout(); plt.show()
    show_channel_grid(acts,f'{layer}: {acts.shape[1]} × {acts.shape[2]} × {len(acts)} (H × W × C)',shared)
    before,after=('Conv1 + ReLU','Pool1') if layer in stage_names[:2] else ('Conv2 + ReLU','Pool2')
    fig,axs=plt.subplots(1,2,figsize=(7,2.8)); lim=max(float(maps[before][0,0].max()),1e-8)
    for ax,name in zip(axs,[before,after]):
        a=maps[name][0,0]; ax.imshow(a,cmap='viridis',vmin=0,vmax=lim,interpolation='nearest')
        ax.set_title(f'{name}, channel 0: {a.shape}'); ax.axis('off')
    plt.tight_layout(); plt.show()
    print('Shapes (H,W,C):',' → '.join(f'{k}: {a.shape[2:]} × {a.shape[1]}' for k,a in maps.items()))

# 描画をまとめて更新するボタン / One button refreshes the complete view
map_epoch=widgets.IntSlider(value=EPOCHS,min=0,max=EPOCHS,description='Epoch',continuous_update=False)
map_shape=widgets.Dropdown(options=[(n,i) for i,n in enumerate(CLASS_NAMES)],description='Shape')
map_example=widgets.IntSlider(value=0,min=0,max=int((yval==0).sum())-1,description='Example',continuous_update=False)
map_layer=widgets.Dropdown(options=stage_names,description='Layer')
map_scale=widgets.Checkbox(value=True,description='Shared color scale')
map_button=widgets.Button(description='Show maps / 表示',button_style='info')
map_out=widgets.Output()
def update_maps(_=None):
    with map_out:
        clear_output(wait=True)
        try: inspect_maps(map_epoch.value,map_shape.value,map_example.value,map_layer.value,map_scale.value)
        except Exception as exc: print(type(exc).__name__,str(exc)); raise
map_button.on_click(update_maps)
display(widgets.VBox([widgets.HBox([map_epoch,map_shape,map_example]),widgets.HBox([map_layer,map_scale,map_button]),map_out]))
update_maps()

## 5. RGBとチャネル間の寄与 / RGB and channel contributions
最初のフィルタは3×3×3。RGBそれぞれの積和を足し、バイアスを加えて１つの応答を作ります。 / A first-layer filter is 3×3×3. RGB contributions sum with a bias to produce one response.

２層目のフィルタは3×3×8。RGBではなく、前の層の８種類の特徴を組み合わせます。 / A second-layer filter is 3×3×8 and combines eight learned features rather than RGB.

フィルタの重みは正負を持ちます。表示される積和を実際の特徴マップと照合します。 / Filter weights are signed. The displayed dot product is checked against the actual feature map.

In [ ]:
def inspect_filter(epoch=EPOCHS,shape=0,example=0,conv=1,channel=0,row=0,col=0):
    idx=select_image(shape,example); p=snapshots[int(epoch)]
    _,maps,_=forward(Xval[idx:idx+1],p)
    inp=maps['Input' if conv==1 else 'Pool1'][0]
    weights=p[f'w{conv}'][channel]; bias=p[f'b{conv}'][channel]
    patch=np.pad(inp,((0,0),(1,1),(1,1)))[:,row:row+3,col:col+3]
    contributions=(patch*weights).sum(axis=(1,2)); z=float(contributions.sum()+bias)
    cols=4; rows=(len(weights)+3)//4
    fig,axs=plt.subplots(rows,cols,figsize=(11,2.8*rows),squeeze=False)
    vmax=max(float(abs(weights).max()),1e-8)
    for c,ax in enumerate(axs.flat):
        if c<len(weights):
            im=ax.imshow(weights[c],cmap='coolwarm',vmin=-vmax,vmax=vmax)
            name=['R','G','B'][c] if conv==1 else f'Feature {c}'
            ax.set_title(f'{name}: contribution {contributions[c]:.4f}')
            fig.colorbar(im,ax=ax,fraction=.046,pad=.04)
        ax.axis('off')
    fig.suptitle(f'Conv{conv}, output channel {channel}, position ({row},{col})'); plt.tight_layout(); plt.show()
    print('各入力チャネルの寄与 / Contributions:',np.round(contributions,4))
    print(f'Sum={contributions.sum():.5f}; bias={bias:.5f}; z={z:.5f}; ReLU(z)={max(0,z):.5f}')
    actual=maps[f'Conv{conv} + ReLU'][0,channel,row,col]
    assert np.isclose(max(0,z),actual,atol=2e-5)

filter_conv=widgets.Dropdown(options=[1,2],description='Conv')
filter_ch=widgets.IntSlider(value=0,min=0,max=7,description='Channel',continuous_update=False)
filter_row=widgets.IntSlider(value=8,min=0,max=15,description='Row',continuous_update=False)
filter_col=widgets.IntSlider(value=8,min=0,max=15,description='Column',continuous_update=False)
filter_button=widgets.Button(description='Show filter / 表示',button_style='info')
filter_out=widgets.Output()
def sync_filter(change=None):
    cv=filter_conv.value
    filter_ch.max=7 if cv==1 else 15
    filter_row.max=filter_col.max=15 if cv==1 else 7
def update_filter(_=None):
    with filter_out:
        clear_output(wait=True)
        inspect_filter(map_epoch.value,map_shape.value,map_example.value,filter_conv.value,filter_ch.value,filter_row.value,filter_col.value)
filter_conv.observe(sync_filter,names='value'); filter_button.on_click(update_filter)
display(widgets.HTML('上のEpoch・Shape・Exampleを使用 / Uses Epoch, Shape, and Example selected above.'))
display(widgets.VBox([widgets.HBox([filter_conv,filter_ch]),widgets.HBox([filter_row,filter_col,filter_button]),filter_out]))
update_filter()

## 6. 同じ図形を移動する / Translate the same shape
局所フィルタを共有するため、入力の移動に応じて特徴マップも移動する傾向があります。 / Shared local filters tend to move their responses with the input.

stride 1の畳み込みは、境界の影響がない位置では整数ピクセルの移動に対応して応答も移動します。ただしPoolの間引きと画像端の処理により完全な移動不変性は保証されません。分類器も含めた予測は変わることがあります。 / Stride-1 convolution is translation equivariant away from boundaries. Pooling, boundaries, and the classifier prevent a guarantee of complete translation invariance.

空いた領域は黒で埋めます。端からはみ出た図形は切れます。この追加操作は訓練データと背景の分布が多少異なります。 / Empty pixels are filled with black. Shapes crossing the boundary are cropped. This operation slightly changes the background distribution relative to training data.

同じサンプル・チャネルの原画像と移動後を、同じ色スケールで比較します。 / Both views use the same image, channel, and color scale.

In [ ]:
def translate_image(image,dx,dy):
    result=np.zeros_like(image); h,w=image.shape[1:]
    sx0=max(0,-dx); sx1=min(w,w-dx); sy0=max(0,-dy); sy1=min(h,h-dy)
    result[:,sy0+dy:sy1+dy,sx0+dx:sx1+dx]=image[:,sy0:sy1,sx0:sx1]
    return result

def compare_translation(shape=0,example=0,dx=2,dy=0,layer='Conv1 + ReLU',channel=0):
    idx=select_image(shape,example); a=Xval[idx]; b=translate_image(a,int(dx),int(dy))
    prob,maps,_=forward(np.stack([a,b]),params)
    channel=int(channel)%maps[layer].shape[1]
    fig,axs=plt.subplots(2,2,figsize=(9,7))
    lim=max(float(maps[layer][:,channel].max()),1e-8)
    for j,image in enumerate([a,b]):
        axs[0,j].imshow(image.transpose(1,2,0),interpolation='nearest')
        axs[0,j].set_title(('Original' if j==0 else f'Shift: dx={dx}, dy={dy}')+f' | Pred: {CLASS_NAMES[prob[j].argmax()]}')
        im=axs[1,j].imshow(maps[layer][j,channel],cmap='viridis',vmin=0,vmax=lim,interpolation='nearest')
        axs[1,j].set_title(f'{layer} | channel {channel}'); fig.colorbar(im,ax=axs[1,j],fraction=.046,pad=.04)
    for ax in axs.flat:ax.axis('off')
    plt.tight_layout(); plt.show()
    print('Probabilities:',CLASS_NAMES,'original=',np.round(prob[0],3),'shifted=',np.round(prob[1],3))
    return prob

shift_layer=widgets.Dropdown(options=stage_names,description='Layer')
shift_channel=widgets.IntSlider(value=0,min=0,max=7,description='Channel',continuous_update=False)
shift_x=widgets.IntSlider(value=2,min=-3,max=3,description='dx',continuous_update=False)
shift_y=widgets.IntSlider(value=0,min=-3,max=3,description='dy',continuous_update=False)
shift_button=widgets.Button(description='Compare / 比較',button_style='info'); shift_out=widgets.Output()
def shift_layer_change(change):
    shift_channel.max=7 if change['new'] in stage_names[:2] else 15
def update_shift(_=None):
    with shift_out:
        clear_output(wait=True)
        compare_translation(map_shape.value,map_example.value,shift_x.value,shift_y.value,shift_layer.value,shift_channel.value)
shift_layer.observe(shift_layer_change,names='value'); shift_button.on_click(update_shift)
display(widgets.HTML('最終学習モデルを使用。Shape・Exampleは上の選択値 / Uses final model and Shape/Example selected above.'))
display(widgets.VBox([widgets.HBox([shift_layer,shift_channel]),widgets.HBox([shift_x,shift_y,shift_button]),shift_out]))
update_shift()

## 7. 独立テスト / Held-out test
設定調整を終えてから最終評価に使います。 / Use for final evaluation after tuning.

In [ ]:
# 最後の重みを独立テストで評価 / Evaluate the final weights on held-out test data
loss,acc=metrics(Xtest,ytest,params)
print(f'Test accuracy: {acc:.2%}; cross-entropy: {loss:.4f}; n={len(ytest)}')
prob=np.concatenate([forward(Xtest[i:i+128],params)[0] for i in range(0,len(Xtest),128)])
ConfusionMatrixDisplay.from_predictions(ytest,prob.argmax(1),cmap='Blues',colorbar=False,display_labels=CLASS_NAMES)
plt.title('Held-out test confusion matrix'); plt.show()

## ボタンが動かない場合 / Widget-free fallback
値を編集し、このセルを再実行してください。 / Edit values and re-run this cell if widgets do not work.

In [ ]:
# 直接関数を呼ぶ / Direct function calls
RUN_DIRECT = False
if RUN_DIRECT:
    inspect_maps(epoch=EPOCHS,shape=0,example=0,layer='Conv2 + ReLU',shared=True)
    inspect_filter(epoch=EPOCHS,shape=0,example=0,conv=1,channel=0,row=8,col=8)
    # 後の移動デモを実行後に利用可能 / Available after running the translation-demo cell
    if 'compare_translation' in globals():
        compare_translation(shape=0,example=0,dx=2,dy=0,layer='Conv1 + ReLU',channel=0)

## 観察ポイント / What to observe
- Epoch 0と学習後の同じ画像を比べる。 / Compare the same image at epoch 0 and after training.
- 異なる図形、同じ図形の異なる色で応答を比較する。 / Compare different shapes and different colors of one shape.
- Poolは縦横を半分にし、チャネル数は保つ。 / Pooling halves spatial dimensions and preserves channel count.
- Conv1はRGBを、Conv2は８つの特徴チャネルを混ぜる。 / Conv1 combines RGB; Conv2 combines eight feature channels.
- 特徴の良さは可視化だけで断定せず、未使用画像の分類性能と合わせて評価する。 / Judge features using held-out performance as well as visualizations.

出典：このノートブックで生成した人工図形。 / Source: synthetic shapes generated in this notebook.
評価対象は同じ生成条件の未使用の人工図形です。実写真への性能を示すものではありません。 / Evaluation uses unseen synthetic shapes from the same generator, not real-world photographs.
